# 소리새김 — wav2vec2 발음형 인식기 미세조정 (+3주차)

**표적(재정의)**: Whisper 프론트엔드를 wav2vec2 발음형 자모 인식기로 교체해 **정발음 오탐률(before 52%)** 을 낮춘다. 학습쌍 = (wav, phones 자모 시퀀스, AI Hub 사람 전사).

- 베이스: `kresnik/wav2vec2-large-xlsr-korean` (한국어 음향 적응 완료 — 2주차 검증 자산)
- CTC 헤드를 자모 vocab(39종)으로 교체 후 미세조정
- 평가: `eval_fp`(오탐률 before 52%) / `eval_det`(감지) — 화자 분리 검증됨

> ⚠️ **데이터 재배포 금지**: AI Hub 데이터는 각자 드라이브에 업로드해 사용. 이 노트북·모델 가중치는 데이터를 포함하지 않는다.

## 0. 런타임 확인 (GPU 필수)

In [ ]:
!nvidia-smi -L

## 1. 의존성 설치
transformers/datasets/torchaudio 스택. Colab 기본 torch에 맞춰 설치.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" torchaudio jiwer evaluate accelerate

## 2. 설정 + 데이터 준비 (zip → 로컬 디스크)
**중요**: 구글 드라이브는 작은 파일 수천 개 I/O가 매우 느리다. wav를 드라이브에서 직접 읽으면 학습이 기어간다. 로컬에서 `experiments/pack_for_colab.py`로 만든 `sorisaegim_colab.zip`(~7.2GB)을 드라이브에 올린 뒤, 아래 셀이 그것을 Colab 로컬 디스크(`/content`)에 풀어 빠른 I/O를 쓴다. 체크포인트만 드라이브에 저장(세션 끊김 대비).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ↓↓↓ 본인 환경에 맞게 수정 ↓↓↓
ZIP_ON_DRIVE = '/content/drive/MyDrive/sorisaegim_colab.zip'  # 업로드한 zip 위치
OUTPUT_DIR   = '/content/drive/MyDrive/sorisaegim/models/w2v2-jamo'  # 체크포인트(드라이브=영구)
DRIVE_ROOT   = '/content/sorisaegim'   # zip 푸는 로컬 경로 (빠름)

import os
BASE_MODEL = 'kresnik/wav2vec2-large-xlsr-korean'
SAMPLE_RATE = 16000
os.makedirs(OUTPUT_DIR, exist_ok=True)

데이터 압축 해제 (세션마다 1회 — /content는 세션 종료 시 사라짐, 재접속 시 이 셀만 재실행).

⚠️ **반드시 Python zipfile로 푼다** (`unzip` 아님). 리눅스 `unzip`은 한글 폴더명(`원천데이터`)을 매니페스트와 다른 유니코드 형태(NFC↔NFD)로 풀어, 파일은 있는데 경로가 안 맞는 문제가 생긴다. zip을 만든 것과 같은 Python으로 풀면 바이트가 정확히 일치.

In [ ]:
import os, time, json, zipfile
SPLITS = os.path.join(DRIVE_ROOT, 'data/aihub/splits')
if not os.path.exists(os.path.join(SPLITS, 'vocab.json')):
    t = time.time()
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    !cp "$ZIP_ON_DRIVE" /content/pkg.zip
    with zipfile.ZipFile('/content/pkg.zip') as z:   # Python 추출 — 한글 경로 정규화 일치
        z.extractall(DRIVE_ROOT)
    print(f'압축 해제 {time.time()-t:.0f}s')
assert os.path.exists(os.path.join(SPLITS, 'vocab.json')), 'zip/경로 확인'
# wav 경로가 매니페스트와 정확히 일치하는지 검증 (unzip 정규화 문제 조기 감지)
_ex = json.loads(open(os.path.join(SPLITS,'val.jsonl'),encoding='utf-8').readline())
assert os.path.exists(os.path.join(DRIVE_ROOT, _ex['wav_path'])), 'wav 경로 불일치 — Python으로 재추출 필요'
print('splits OK + wav 경로 일치:', SPLITS)

## 3. 프로세서 (토크나이저 + 특징추출기)
라벨은 개별 자모 문자열(공백 제거). 각 자모가 vocab 토큰 1개 → CTC 타깃.

In [ ]:
import json
from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor)

# vocab.json: {'[PAD]':0, '[UNK]':1, 'ㄱ':2, ...}. PAD=0=CTC blank.
tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(SPLITS, 'vocab.json'),
    unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token=None)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SAMPLE_RATE, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print('vocab size:', len(tokenizer))

## 4. 데이터셋 로드
jsonl → datasets. 오디오는 학습 중 지연 로딩(torchaudio, 16k 리샘플).
`label`(실발화 phones)을 CTC 타깃으로, `expected`(제시어 기대발음)는 평가용으로 보존.

In [ ]:
from datasets import load_dataset, Audio

def jsonl(name):
    return os.path.join(SPLITS, name)

ds = load_dataset('json', data_files={
    'train': jsonl('train.jsonl'), 'val': jsonl('val.jsonl'),
    'eval_fp': jsonl('eval_fp.jsonl'), 'eval_det': jsonl('eval_det.jsonl'),
})

def abspath(batch):
    batch['audio'] = os.path.join(DRIVE_ROOT, batch['wav_path'])
    return batch
ds = ds.map(abspath)
ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
print(ds)

## 5. 전처리 — 오디오 특징 + 라벨 인코딩

In [ ]:
def prepare(batch):
    audio = batch['audio']
    batch['input_values'] = processor(
        audio['array'], sampling_rate=SAMPLE_RATE).input_values[0]
    batch['input_length'] = len(batch['input_values'])
    # 공백 제거 → 연속 자모 문자열. 각 자모가 토큰 1개.
    text = ''.join(batch['label'].split())
    batch['labels'] = processor(text=text).input_ids
    return batch

train_ds = ds['train'].map(prepare, remove_columns=ds['train'].column_names,
                           num_proc=2)
val_ds = ds['val'].map(prepare, remove_columns=ds['val'].column_names, num_proc=2)

### 5b. 긴 클립 필터 (OOM 예방)
발화 대부분은 짧지만 일부 문장이 ~10초라 메모리 피크를 키운다. 12초 초과만 제거(극소수).

In [ ]:
MAX_SEC = 12
n0 = (train_ds.num_rows, val_ds.num_rows)
train_ds = train_ds.filter(lambda L: L <= SAMPLE_RATE*MAX_SEC, input_columns=['input_length'])
val_ds   = val_ds.filter(lambda L: L <= SAMPLE_RATE*MAX_SEC, input_columns=['input_length'])
print('길이 필터:', n0, '→', (train_ds.num_rows, val_ds.num_rows))

## 6. 데이터 콜레이터 (CTC 패딩)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTC:
    processor: Wav2Vec2Processor
    def __call__(self, features):
        input_features = [{'input_values': f['input_values']} for f in features]
        label_features = [{'input_ids': f['labels']} for f in features]
        batch = self.processor.pad(input_features, padding=True, return_tensors='pt')
        labels_batch = self.processor.pad(
            labels=label_features, padding=True, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        batch['labels'] = labels
        return batch

data_collator = DataCollatorCTC(processor=processor)

## 7. 지표 — CER (val)

In [ ]:
import evaluate
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = pred_logits.argmax(-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    # 공백 제거 후 CER (자모 문자 단위)
    cer = cer_metric.compute(predictions=[p.replace(' ','') for p in pred_str],
                             references=[l.replace(' ','') for l in label_str])
    return {'cer': cer}

## 8. 모델 — CTC 헤드 교체 + 특징인코더 동결
베이스의 한국어 vocab 헤드를 자모 39종 헤드로 교체(`ignore_mismatched_sizes`). CNN 특징인코더는 동결(적은 데이터에 안정적).

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction='mean',
    ignore_mismatched_sizes=True,   # 헤드 크기 불일치 → 새 헤드 랜덤 초기화
)
model.freeze_feature_encoder()
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

## 9. 학습 설정 + Trainer
T4(15GB)에서 wav2vec2-large는 긴 오디오에서 활성값이 커 OOM 위험 — 배치 2 + 누적 8(유효 배치 16) + 5b 긴 클립 필터로 안정화. OOM 재발 시 배치 1 / 누적 16.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # 단편화 완화
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,     # 유효 배치 16
    per_device_eval_batch_size=2,
    num_train_epochs=8,
    learning_rate=3e-4,        # 새 헤드 위주 — 필요시 1e-4로 낮춤(2차 학습)
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    save_total_limit=2,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=data_collator, compute_metrics=compute_metrics,
    processing_class=processor,   # 최신 transformers: tokenizer= → processing_class=
)

## 10. 학습
체크포인트는 드라이브에 저장 → Colab 세션 끊겨도 재개 가능(`resume_from_checkpoint=True`).

In [ ]:
trainer.train()   # 재개: trainer.train(resume_from_checkpoint=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

## 11. 핵심 평가 — 오탐률(before 52%) vs 감지율
모델이 출력한 발음형을 **제시어 기대발음(expected)** 과 비교. 자모 1개라도 다르면 '오류 플래그'.
- `eval_fp`(정발음): 플래그율 = **오탐률** → Whisper 52% 대비 개선 확인
- `eval_det`(오류): 플래그율 = **감지율**
목표: 오탐률을 52%보다 크게 낮추면서 감지율 유지.

> ⚠️ **오탐률 바닥은 ~12%**: 정발음에서도 phones 정본과 기대발음(expected)이 87.8%만 완전일치(작업2)한다. 나머지 ~12%는 화자의 미세 억양·전사 입도차라, 완벽한 모델이라도 이만큼은 '다름'으로 뜬다. 따라서 52%→12% 근처가 이론적 상한 개선. 이 바닥을 더 낮추려면 플래그 임계를 편집거리≥2로 완화(12번 힌트).

In [ ]:
import torch, numpy as np

def normalize(s):
    return s.replace(' ', '')

@torch.no_grad()
def transcribe_jamo(audio_array):
    inputs = processor(audio_array, sampling_rate=SAMPLE_RATE, return_tensors='pt')
    logits = model(inputs.input_values.to(model.device)).logits
    pred_ids = logits.argmax(-1)
    return normalize(processor.batch_decode(pred_ids)[0])

def flag_rate(split):
    n, flagged = 0, 0
    for ex in ds[split]:
        pred = transcribe_jamo(ex['audio']['array'])
        expected = normalize(ex['expected'])
        if pred != expected:   # 자모 1개라도 다르면 오류 플래그
            flagged += 1
        n += 1
    return flagged / n, n

model.eval()
fp_rate, n_fp = flag_rate('eval_fp')
det_rate, n_det = flag_rate('eval_det')
print(f'오탐률(eval_fp {n_fp}건): {fp_rate*100:.0f}%   (Whisper before 52%)')
print(f'감지율(eval_det {n_det}건): {det_rate*100:.0f}%')
print(f'순개선: 오탐 {52 - fp_rate*100:+.0f}%p')

## 12. 다음 (2차 학습 힌트)
- 오탐률이 52%보다 낮으면 → 표적 달성. `eval_det` 규칙별 세부 감지로 약점 확인.
- CER은 좋은데 오탐이 안 줄면 → 플래그 임계를 '자모 1개'에서 '편집거리 ≥2'로 완화(엔진 측 튜닝).
- 학습 불안정(CER 발산) → learning_rate 1e-4로 낮추고 특징인코더 동결 유지.
- 개선폭이 작으면 → Zeroth-Korean(원어민 정발음)으로 정발음 표본 보강 후 재학습.
- **한 번에 하나만 바꾸고 기록** — ablation 표가 이력서 근거가 된다.